In [4]:
import torch
import vllm

print("vLLM:", vllm.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

vLLM: 0.31.0
PyTorch: 2.13.0+cu130
CUDA available: True
GPU: Tesla T4


--------------------------------------------------------------------------------

vLLM loads the model onto the available GPU, starts an API server, and waits for clients to send prompts. It then generates responses and returns them through the API.

In [1]:
!vllm serve Qwen/Qwen2.5-0.5B-Instruct \
    --host 0.0.0.0 \
    --port 8000 \
    --api-key day1-test-key

(APIServer pid=5031) INFO 10-15 16:44:41 [api_utils.py:347] 
(APIServer pid=5031) INFO 10-15 16:44:41 [api_utils.py:347]        █     █     █▄   ▄█
(APIServer pid=5031) INFO 10-15 16:44:41 [api_utils.py:347]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.31.0
(APIServer pid=5031) INFO 10-15 16:44:41 [api_utils.py:347]   █▄█▀ █     █     █     █  model   Qwen/Qwen2.5-0.5B-Instruct
(APIServer pid=5031) INFO 10-15 16:44:41 [api_utils.py:347]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=5031) INFO 10-15 16:44:41 [api_utils.py:347] 
(APIServer pid=5031) INFO 10-15 16:44:41 [api_utils.py:286] non-default args: {'model_tag': 'Qwen/Qwen2.5-0.5B-Instruct', 'host': '0.0.0.0', 'api_key': '***', 'model': 'Qwen/Qwen2.5-0.5B-Instruct'}
config.json: 100% 659/659 [00:00<00:00, 3.84MB/s]
(APIServer pid=5031) INFO 10-15 16:45:01 [model.py:699] Resolved architecture: Qwen2ForCausalLM
(APIServer pid=5031) WARNING 10-15 16:45:01 [model.py:2389] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch

In [6]:
import subprocess
import time
import requests

with open("/content/vllm-server.log", "w") as log:
    process = subprocess.Popen(
        [
            "vllm", "serve", "Qwen/Qwen2.5-0.5B-Instruct",
            "--host", "0.0.0.0",
            "--port", "8000",
            "--api-key", "day1-test-key"
        ],
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True
    )

print("Server PID:", process.pid)

Server PID: 9835


In [7]:
import time
import requests

for _ in range(90):
    try:
        response = requests.get(
            "http://127.0.0.1:8000/health",
            timeout=2
        )
        if response.status_code == 200:
            print("vLLM server is ready!")
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    print("Server did not become ready.")

!tail -n 50 /content/vllm-server.log

vLLM server is ready!
(EngineCore pid=9991) INFO 10-15 17:04:11 [model_runner.py:1087] Graph capturing finished in 4 secs, took 0.24 GiB
(EngineCore pid=9991) INFO 10-15 17:04:11 [gpu_worker.py:692] Available KV cache memory: 11.83 GiB
(EngineCore pid=9991) INFO 10-15 17:04:11 [gpu_worker.py:707] CUDA graph memory profiling is enabled (default since v0.21.0). The current --gpu-memory-utilization=0.9200 is equivalent to --gpu-memory-utilization=0.9010 without CUDA graph memory profiling. To maintain the same effective KV cache size as before, increase --gpu-memory-utilization to 0.9390. To disable, set VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS=0.
(EngineCore pid=9991) INFO 10-15 17:04:11 [kv_cache_utils.py:2464] GPU KV cache size: 1,033,664 tokens, Maximum concurrency for 32,768 tokens per request: 31.54x
(EngineCore pid=9991) INFO 10-15 17:04:11 [kernel_warmup.py:172] JIT kernel warmup starting.
(EngineCore pid=9991) INFO 10-15 17:04:11 [kernel_warmup.py:185] JIT kernel warmup finished 

In [9]:
import requests

response = requests.post(
    "http://localhost:8000/v1/chat/completions",
    headers={"Authorization": "Bearer day1-test-key"},
    json={
        "model": "Qwen/Qwen2.5-0.5B-Instruct",
        "messages": [
            {
                "role": "user",
                "content": "who is narendra modi"
            }
        ],
        "max_tokens": 100
    },
    timeout=120
)

print("Status:", response.status_code)
print(response.json()["choices"][0]["message"]["content"])

Status: 200
Narendra Modi is the current Prime Minister of India and the leader of the Bharatiya Janata Party (BJP). He has been in power since 2014 and has held several high-ranking positions in government, including President of India, Chairman of the Reserve Bank of India, and former Governor of India.

Modi is known for his commitment to economic growth, social welfare programs, and reforming the Indian economy through privatization and deregulation. He has also been involved in various
